#### واردات کتابخانه ها

In [1]:
import pandas as pd
import numpy as np
from hazm import Normalizer, WordTokenizer, stopwords_list
import re
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import FastText
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

#### بارگذاری دیتاست (آموزشی)

In [2]:
df = pd.read_parquet("./data/sentiment_dksf_train.parquet")

In [3]:
print("ستون‌های دیتاست:", df.columns.tolist())

ستون‌های دیتاست: ['text', 'label']


In [4]:
print("توزیع کلاس های ستون\n", df['label'].value_counts())

توزیع کلاس های ستون
 label
0    10206
1     9587
2     8809
Name: count, dtype: int64


In [5]:
print("تعداد نمونه‌ها:", len(df))

تعداد نمونه‌ها: 28602


In [6]:
print("نمونه‌های اولیه:")
print(df.head(5))

نمونه‌های اولیه:
                                                text  label
0                         شیشه هاش همون روز اول شکست      0
1  تمیز -مرتب- چقدر پیک آن تایم و مؤدب …سپاس بسیا...      1
2  نظر من هیچ گاه این محصول رو خریداری نکنید اگه ...      0
3  برنج ایرانی سفارش دادم افتضاح بود با هندی هیچ ...      0
4         سلام اصلا راضی نیستم  یعنی ارزش خرید نداره      0


#### آمار اولیه قبل از پیش پردازش

In [7]:
def text_stats(df, text_col='text'):
    lengths_char = df[text_col].str.len()
    lengths_word = df[text_col].apply(lambda x: len(x.split()))
    unique_words = len(set(' '.join(df[text_col].astype(str)).split()))
    
    print(f"میانگین طول متن (کاراکتر): {lengths_char.mean():.2f}")
    print(f"میانگین طول متن (کلمه): {lengths_word.mean():.2f}")
    print(f"تعداد کلمات منحصربه‌فرد: {unique_words}")

print("آمار متن خام (قبل از پیش پردازش) :")
text_stats(df)

آمار متن خام (قبل از پیش پردازش) :
میانگین طول متن (کاراکتر): 125.53
میانگین طول متن (کلمه): 25.77
تعداد کلمات منحصربه‌فرد: 58975


#### پیش پردازش متن

In [8]:
normalizer = Normalizer()

tokenizer = WordTokenizer()

custom_stopwords = {
    # حروف اضافه و ربط خیلی رایج
    'از', 'به', 'در', 'با', 'برای', 'تا', 'را', 'و', 'که', 'این', 'اون', 'آن', 'های', 'ها', 'ای', 'ام', 'ات', 'اش', 'شان', 'تان', 'مان',

    # افعال کمکی و بودن/شدن
    'است', 'بود', 'بودم', 'بودی', 'بودیم', 'بودند', 'هست', 'هستم', 'هستی', 'هستیم', 'هستند', 'شد', 'شدم', 'شدی', 'شدیم', 'شدند',

    # کلمات خیلی عمومی بدون بار احساسی قوی
    'می', 'میشه', 'میتونه', 'میتونم', 'میتونی', 'میکنه', 'میکنم', 'میکنی', 'میکنیم', 'میکنن',
    'یه', 'ی', 'هم', 'نیز', 'یا', 'ولی', 'اما', 'اگر', 'تا', 'پس', 'چون', 'هر', 'همه', 'هیچ', 'چند', 'بسیار', 'خیلی', 'کمی',

    # ضمایر و کلمات اشاره‌ای خیلی رایج
    'من', 'تو', 'او', 'ما', 'شما', 'ایشان', 'خود', 'خودم', 'خودت', 'خودش', 'خودمون', 'خودتون',

    # برخی کلمات ساختاری
    'بودن', 'شدن', 'کردن', 'داشتن', 'داشتم', 'داشتی', 'داشتیم', 'داشتند'
}

stopwords = custom_stopwords

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s\u0600-\u06FF]', '', text)
    text = normalizer.normalize(text)
    text = text.replace('\u200c', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_text(text):
    cleaned = clean_text(text)
    if not cleaned:
        return []
    tokens = tokenizer.tokenize(cleaned)
    tokens = [
        t for t in tokens
        if t not in stopwords
        and len(t) > 1
        and not t.isdigit()
    ]
    return tokens

In [9]:
print("...در حال پیش پردازش متن‌ها (ممکن است چند دقیقه طول بکشه)")
df['clean_text'] = df['text'].apply(clean_text)
df['tokens'] = df['clean_text'].apply(preprocess_text)
df['clean_text_joined'] = df['tokens'].apply(lambda x: ' '.join(x) if x else '')

...در حال پیش پردازش متن‌ها (ممکن است چند دقیقه طول بکشه)


#### آمار بعد از پیش پردازش

In [10]:
print("آمار متن بعد از پیش پردازش :")
text_stats(df, text_col='clean_text_joined')

all_words_after = ' '.join(df['clean_text_joined'])
unique_after = len(set(all_words_after.split()))
print(f"تعداد کلمات منحصربه‌فرد بعد از پیش پردازش : {unique_after}")

آمار متن بعد از پیش پردازش :
میانگین طول متن (کاراکتر): 98.21
میانگین طول متن (کلمه): 18.31
تعداد کلمات منحصربه‌فرد: 43449
تعداد کلمات منحصربه‌فرد بعد از پیش پردازش : 43449


In [11]:
print("نمونه های قبل و بعد پیش پردازش :")
for i in range(5):
    print(f"\nنمونه {i+1} :")
    print("خام:", df['text'].iloc[i])
    print("پاک شده :", df['clean_text'].iloc[i])
    print("توکن‌ها :", df['tokens'].iloc[i][:15], "...")

نمونه های قبل و بعد پیش پردازش :

نمونه 1 :
خام: شیشه هاش همون روز اول شکست
پاک شده : شیشه هاش همون روز اول شکست
توکن‌ها : ['شیشه', 'هاش', 'همون', 'روز', 'اول', 'شکست'] ...

نمونه 2 :
خام: تمیز -مرتب- چقدر پیک آن تایم و مؤدب …سپاس بسیار عالی فقط کاشکی رنگ دستمال همونی می‌بود که در شکل انتخاب کردیم … باز هم سپاس بسیار عالی از سرعت و کیفیت
پاک شده : تمیز مرتب چقدر پیک آن تایم و مؤدب سپاس بسیار عالی فقط کاشکی رنگ دستمال همونی می بود که در شکل انتخاب کردیم باز هم سپاس بسیار عالی از سرعت و کیفیت
توکن‌ها : ['تمیز', 'مرتب', 'چقدر', 'پیک', 'تایم', 'مؤدب', 'سپاس', 'عالی', 'فقط', 'کاشکی', 'رنگ', 'دستمال', 'همونی', 'شکل', 'انتخاب'] ...

نمونه 3 :
خام: نظر من هیچ گاه این محصول رو خریداری نکنید اگه می خواهید شانستون رو امتحان کنین ببینین کارت هاتون 108 تا یا 52 هستش راه باز هست بعضی ها میگن 108 کارت داره بعضی ها هم میگن 54 تا کارت داره در اصل این محصول باید 108 کارت باشه که برای من 52 تا دراومد و فارسی هم هستش و برای یک کالای دیگه هستش که قیمتش ارزونتره
پاک شده : نظر من هیچ گاه این محصول رو خریداری

In [12]:
df.to_csv("./data/sentiment_processed.csv", index=False, encoding='utf-8-sig')
print(".انجام شد sentiment_processed.csv ذخیره")

.انجام شد sentiment_processed.csv ذخیره


In [13]:
df = pd.read_csv("./data/sentiment_processed.csv")

In [14]:
print("ستون‌ها:", df.columns.tolist())

ستون‌ها: ['text', 'label', 'clean_text', 'tokens', 'clean_text_joined']


#### نمایش متن

<div align="center">TF-IDF</div>

In [15]:
vectorizer = TfidfVectorizer(max_features=5000, min_df=2, analyzer='word')
tfidf_embeddings = vectorizer.fit_transform(df['clean_text_joined'])

print("TF-IDF Embeddings :")
print(f"ابعاد ماتریس : {tfidf_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(vectorizer.vocabulary_)}")
print(f"برای 10 متن اول embedding نمونه : {tfidf_embeddings[0].toarray()[0][:10]}...")

all_words = ' '.join(df['clean_text_joined']).split()
unique_words = set(all_words)
tfidf_coverage = len(vectorizer.vocabulary_) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : {tfidf_coverage:.2f}%")

ValueError: np.nan is an invalid document, expected byte or unicode string.

#### NaN بررسی تعداد داده های

In [16]:
print("clean_text_joined در NaN تعداد :\n", df['clean_text_joined'].isna().sum())
print("تعداد رشته خالی (''):", (df['clean_text_joined'] == '').sum())

problematic = df[df['clean_text_joined'].isna() | (df['clean_text_joined'] == '')]
print("\nنمونه ردیف‌های مشکل‌دار :")
print(problematic[['text', 'clean_text', 'clean_text_joined']].head())

print("\nتعداد کل ردیف‌های مشکل‌دار :", len(problematic))

clean_text_joined در NaN تعداد :
 161
تعداد رشته خالی (''): 0

نمونه ردیف‌های مشکل‌دار :
              text clean_text clean_text_joined
79   .............        NaN               NaN
255           ۴و ۳      ۴ و ۳               NaN
360         ۲٫ ۳٫۸  ۲ ٫ ۳ ٫ ۸               NaN
490           و۲و۸    و ۲ و ۸               NaN
775             --        NaN               NaN

تعداد کل ردیف‌های مشکل‌دار : 161


#### NaN حذف داده های

In [17]:
print("قبل از حذف :")
print("NaN تعداد :", df['clean_text_joined'].isna().sum())
print("تعداد کل نمونه‌ها قبل :", len(df))

df = df.dropna(subset=['clean_text_joined']).reset_index(drop=True)

print("\nبعد از حذف :")
print("تعداد نمونه باقی‌مانده :", len(df))
print("NaN تعداد :", df['clean_text_joined'].isna().sum())

print("\nبررسی نهایی :")
print("تعداد رشته خالی باقی‌مانده :", (df['clean_text_joined'] == '').sum())
print("strip حداقل طول متن بعد از :")
print(df['clean_text_joined'].str.strip().str.len().min())

قبل از حذف :
NaN تعداد : 161
تعداد کل نمونه‌ها قبل : 28602

بعد از حذف :
تعداد نمونه باقی‌مانده : 28441
NaN تعداد : 0

بررسی نهایی :
تعداد رشته خالی باقی‌مانده : 0
strip حداقل طول متن بعد از :
2


#### تصمیم گیری برای بررسی تعداد ویژگی ها

In [18]:
for mf in [5000, 10000, 15000, None]:
    print(f"\n--- max_features = {mf} ---")
    vec = TfidfVectorizer(max_features=mf, min_df=2, analyzer='word')
    emb = vec.fit_transform(df['clean_text_joined'])
    vocab_size = len(vec.vocabulary_)
    all_unique = len(set(' '.join(df['clean_text_joined']).split()))
    coverage = vocab_size / all_unique * 100 if all_unique else 0
    print(f"ویژگی‌ها: {vocab_size:,} | پوشش: %{coverage:.2f}")
    print(f"ابعاد: {emb.shape}")


--- max_features = 5000 ---
ویژگی‌ها: 5,000 | پوشش: %11.51
ابعاد: (28441, 5000)

--- max_features = 10000 ---
ویژگی‌ها: 10,000 | پوشش: %23.02
ابعاد: (28441, 10000)

--- max_features = 15000 ---
ویژگی‌ها: 15,000 | پوشش: %34.52
ابعاد: (28441, 15000)

--- max_features = None ---
ویژگی‌ها: 16,307 | پوشش: %37.53
ابعاد: (28441, 16307)


In [19]:
vectorizer = TfidfVectorizer(max_features=15000, min_df=2, analyzer='word')
tfidf_embeddings = vectorizer.fit_transform(df['clean_text_joined'])

print("TF-IDF Embeddings :")
print(f"ابعاد ماتریس : {tfidf_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(vectorizer.vocabulary_)}")
print("برای 10 متن اول embedding نمونه :")
print(tfidf_embeddings[0].toarray()[0][:10])

all_words = ' '.join(df['clean_text_joined']).split()
unique_words = set(all_words)
tfidf_coverage = len(vectorizer.vocabulary_) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{tfidf_coverage:.2f}")

TF-IDF Embeddings :
ابعاد ماتریس : (28441, 15000)
تعداد واژگان پوشش داده‌شده : 15000
برای 10 متن اول embedding نمونه :
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
پوشش واژگان : %34.52


In [21]:
np.save(r"D:\data-Project-NLP\tfidf_embeddings.npy", tfidf_embeddings.toarray())
print("\n.انجام شد tfidf_embeddings.npy ذخیره")


.انجام شد tfidf_embeddings.npy ذخیره


<div align="center">Fast Text</div>

In [22]:
sentences = df['tokens'].tolist()

fasttext_model = FastText(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=5,
    epochs=20,
    sg=1,
    workers=4
)

def get_fasttext_embedding(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

fasttext_embeddings = np.array([get_fasttext_embedding(tokens, fasttext_model) for tokens in df['tokens']])

print("FastText Embeddings :")
print(f"ابعاد هر بردار : {fasttext_model.vector_size}")
print(f"ابعاد ماتریس : {fasttext_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(fasttext_model.wv)}")
print("برای متن اول embedding نمونه :")
print(fasttext_embeddings[0][:10])

fasttext_coverage = len(fasttext_model.wv) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{fasttext_coverage:.2f}")

FastText Embeddings :
ابعاد هر بردار : 300
ابعاد ماتریس : (28441, 300)
تعداد واژگان پوشش داده‌شده : 104
برای متن اول embedding نمونه :
[ 0.0783065  -0.0217669   0.0196003  -0.01712447  0.00258109 -0.021798
  0.0029087  -0.02642851 -0.05190584 -0.01168475]
پوشش واژگان : %0.24


In [23]:
sentences = df['tokens'].tolist()

fasttext_model = FastText(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=2,
    epochs=25,
    sg=1,
    workers=4
)

def get_fasttext_embedding(tokens, model):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

fasttext_embeddings = np.array([get_fasttext_embedding(tokens, fasttext_model) for tokens in df['tokens']])

print("FastText Embeddings :")
print(f"ابعاد هر بردار : {fasttext_model.vector_size}")
print(f"ابعاد ماتریس : {fasttext_embeddings.shape}")
print(f"تعداد واژگان پوشش داده‌شده : {len(fasttext_model.wv)}")
print("برای متن اول embedding نمونه :")
print(fasttext_embeddings[0][:10])

fasttext_coverage = len(fasttext_model.wv) / len(unique_words) * 100 if unique_words else 0
print(f"پوشش واژگان : %{fasttext_coverage:.2f}")

FastText Embeddings :
ابعاد هر بردار : 300
ابعاد ماتریس : (28441, 300)
تعداد واژگان پوشش داده‌شده : 107
برای متن اول embedding نمونه :
[-0.00805286 -0.06531681 -0.02056436  0.00834208 -0.00018374  0.04808907
  0.01328954 -0.02957014  0.01472696  0.03237038]
پوشش واژگان : %0.25


In [17]:
print("آمار طول لیست توکن‌ها :")
print(df['tokens'].apply(len).describe())

print("\nخالی است tokens تعداد ردیف‌هایی که :")
print((df['tokens'].apply(len) == 0).sum())

print("\nفقط یک کلمه دارد tokens تعداد ردیف‌هایی که :")
print((df['tokens'].apply(len) == 1).sum())

آمار طول لیست توکن‌ها :
count    28447.000000
mean       134.786199
std        162.067961
min          5.000000
25%         45.000000
50%         90.000000
75%        168.000000
max       3179.000000
Name: tokens, dtype: float64

خالی است tokens تعداد ردیف‌هایی که :
0

فقط یک کلمه دارد tokens تعداد ردیف‌هایی که :
0


In [18]:
print("tokens نمونه ۵ ردیف اول :")
for i in range(5):
    print(f"ردیف {i}: طول = {len(df['tokens'].iloc[i])}, توکن‌ها = {df['tokens'].iloc[i]}")

print("\nنمونه ردیفی که طول توکن > 0 داره (اولین مورد) :")
non_empty = df[df['tokens'].apply(len) > 0]
if not non_empty.empty:
    print(non_empty.iloc[0][['clean_text_joined', 'tokens']])
else:
    print("!هیچ ردیفی توکن ندارد")

tokens نمونه ۵ ردیف اول :
ردیف 0: طول = 38, توکن‌ها = ['شیشه', 'هاش', 'همون', 'روز', 'شکست']
ردیف 1: طول = 141, توکن‌ها = ['تمیز', 'مرتب', 'چقدر', 'پیک', 'تایم', 'مؤدب', 'سپاس', 'کاشکی', 'رنگ', 'دستمال', 'همونی', 'شکل', 'انتخاب', 'کردیم', 'سپاس', 'سرعت', 'کیفیت']
ردیف 2: طول = 318, توکن‌ها = ['محصول', 'خریداری', 'نکنید', 'اگه', 'خواهید_شانستون', 'امتحان', 'کنین', 'ببینین', 'کارت', 'هاتون', '۱۰۸', '۵۲', 'هستش', 'هست', 'ها', 'میگن', '۱۰۸', 'کارت', 'داره', 'ها', 'میگن', '۵۴', 'کارت', 'داره', 'اصل', 'محصول', '۱۰۸', 'کارت', 'باشه', '۵۲', 'دراومد', 'فارسی', 'هستش', 'کالای', 'دیگه', 'هستش', 'قیمتش', 'ارزونتره']
ردیف 3: طول = 68, توکن‌ها = ['برنج', 'ایرانی', 'سفارش', 'دادم', 'افتضاح', 'هندی', 'فرقی', 'نمی']
ردیف 4: طول = 58, توکن‌ها = ['سلام', 'اصلا', 'راضی', 'نیستم', 'ارزش', 'خرید', 'نداره']

نمونه ردیفی که طول توکن > 0 داره (اولین مورد) :
clean_text_joined                    شیشه هاش همون روز شکست
tokens               ['شیشه', 'هاش', 'همون', 'روز', 'شکست']
Name: 0, dtype: object


In [ ]:
all_tokens = [t for sublist in df['tokens'] for t in sublist]
word_counts = Counter(all_tokens)

print("تعداد کل توکن‌های منحصربه‌فرد قبل از آموزش :", len(word_counts))
print("۱۰ واژه پرتکرار :", word_counts.most_common(10))
print("تعداد واژه‌هایی که دقیقاً ۱ بار ظاهر شدن :", sum(1 for count in word_counts.values() if count == 1))
print("تعداد واژه‌هایی که حداقل ۲ بار ظاهر شدن :", sum(1 for count in word_counts.values() if count >= 2))

تعداد کل توکن‌های منحصربه‌فرد قبل از آموزش : 128
۱۰ واژه پرتکرار : [("'", 910058), (',', 426582), (' ', 426582), ('ا', 237747), ('ی', 199366), ('ر', 140805), ('م', 129058), ('ن', 127190), ('ه', 122213), ('د', 120621)]
تعداد واژه‌هایی که دقیقاً ۱ بار ظاهر شدن : 9
تعداد واژه‌هایی که حداقل ۲ بار ظاهر شدن : 119


In [ ]:
# np.save(r"D:\data-Project-NLP\fasttext_embeddings.npy", fasttext_embeddings)
# print("\n.انجام شد fasttext_embeddings.npy ذخیره")

<div align="center">ParsBERT</div>

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"استفاده از دستگاه : {device}")

# model_name = "HooshvareLab/bert-fa-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name).to(device)

# def get_parsbert_embeddings_batch(texts, tokenizer, model, batch_size=32, max_length=128):
#     embeddings = []
#     model.eval()
#     with torch.no_grad():
#         for i in range(0, len(texts), batch_size):
#             batch = texts[i:i+batch_size]
#             inputs = tokenizer(batch, return_tensors="pt", max_length=max_length, 
#                              truncation=True, padding=True)
#             inputs = {k: v.to(device) for k, v in inputs.items()}
#             outputs = model(**inputs)
#             batch_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
#             embeddings.append(batch_emb)
#             print(f"پردازش batch {i//batch_size + 1} / {len(texts)//batch_size + 1}")
#     return np.vstack(embeddings)

# print("...(ممکن است چند دقیقه طول بکشد) ParsBERT embeddings در حال محاسبه")
# parsbert_embeddings = get_parsbert_embeddings_batch(df['clean_text_joined'].tolist(), tokenizer, model)

# print("\nParsBERT Embeddings :")
# print(f"ابعاد هر  بردار : {parsbert_embeddings.shape[1]}")
# print(f"ابعاد ماتریس : {parsbert_embeddings.shape}")
# print("tokenizer اندازه واژگان :")
# print(len(tokenizer.vocab))
# print("برای متن اول embedding نمونه :")
# print(parsbert_embeddings[0][:10])

# coverage_pb = 100.0
# print(f"پوشش واژگان تقریبی : {coverage_pb:.2f}%")

In [ ]:
# np.save(r"D:\data-Project-NLP\parsbert_embeddings.npy", parsbert_embeddings)
# print("\n.انجام شد parsbert_embeddings.npy ذخیره")

#### مقایسه روش ها

In [ ]:
# def compute_similarity(emb1, emb2):
#     return cosine_similarity([emb1], [emb2])[0][0]

# sim_tfidf = compute_similarity(tfidf_embeddings[0].toarray()[0], tfidf_embeddings[1].toarray()[0])
# sim_ft    = compute_similarity(fasttext_embeddings[0], fasttext_embeddings[1])
# sim_pb    = compute_similarity(parsbert_embeddings[0], parsbert_embeddings[1])

# print("مقایسه روش‌ها :")
# comparison = pd.DataFrame({
#     "روش": ["TF-IDF", "FastText", "ParsBERT"],
#     "ابعاد": [tfidf_embeddings.shape[1], fasttext_model.vector_size, parsbert_embeddings.shape[1]],
#     "پوشش واژگان (%)": [tfidf_coverage, fasttext_coverage, 100.0],
#     "پارامترهای کلیدی": ["max_features=5000, min_df=2", "vector_size=300, min_count=5, epochs=20", "768 dim, mean pooling"],
#     "1 vs 0 متن similarity": [sim_tfidf, sim_ft, sim_pb]
# })
# print(comparison)